# 🪆 10.13 Complex Data Types & Nested Data

Welcome back! In the last lesson (**10.12 Working with Strings & Dates**), we learned how to clean up messy text and dates.

Up until now, every DataFrame we've worked with has been **"Flat"**. This means every cell in our table contained exactly one simple value: one name, one age, or one date.

But what if you are pulling data from a modern web application, like a food delivery app API? The data isn't flat. A single user might have a *list* of 10 previous orders packed into one single column! Or, they might have an "Address" column that secretly contains a nested object with street, city, and zip code inside it.

In this lesson, we will learn how to handle these **Complex Data Types** (Arrays, Maps, and Structs) and how to "flatten" them out so our Data Scientists can actually use them.

### 🎯 Learning Objectives
* Understand the three complex data types: **ArrayType**, **MapType**, and **StructType**.
* Learn how to navigate and extract data from nested JSON objects using the dot notation (`col("struct.field")`).
* Master the `explode()` function to turn lists into individual rows.
* Use `posexplode()` to track the position of items in a list.
* Understand why "flattening" data is a core responsibility of a Data Engineer.

In [ ]:
# 0. Setup: Let's start our SparkSession and create some highly nested JSON data.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os

spark = SparkSession.builder.appName("Complex_Data_Types").master("local[*]").getOrCreate()

# Let's pretend we hit an API for an e-commerce store and got this JSON back.
# Notice how "address" has sub-fields, and "purchases" is a list of items!
nested_json_content = """
{"user_id": 1, "name": "Alice", "address": {"city": "Seattle", "zip": "98101"}, "purchases": ["Laptop", "Mouse", "Monitor"]}
{"user_id": 2, "name": "Bob", "address": {"city": "Austin", "zip": "98102"}, "purchases": ["Keyboard"]}
{"user_id": 3, "name": "Charlie", "address": {"city": "Denver", "zip": "78701"}, "purchases": ["Webcam", "Desk"]}
"""

with open("nested_api_data.json", "w") as f:
    f.write(nested_json_content.strip())

# Read the JSON file
df_nested = spark.read.json("nested_api_data.json")

print("Raw Nested DataFrame:")
df_nested.show(truncate=False)

# Look closely at the schema output!
print("The Schema (Notice the Complex Types):")
df_nested.printSchema()

## 1. The Three Complex Data Types

When you looked at the schema above, you probably saw some new words. PySpark has three "Complex" Data Types for handling nested data:

### 📦 1. `ArrayType` (Lists)
An Array is an ordered list of items, where every item is the *same* data type (e.g., a list of strings).
* **Analogy:** A shopping cart. It contains many individual items (`["Laptop", "Mouse"]`), but it's carried around as one single object.
* **In our schema:** The `purchases` column is an `ArrayType(StringType)`. 

### 🪆 2. `StructType` (Nested Objects)
A Struct is an object that contains its own internal columns (fields), which can be *different* data types. It's literally a table inside a column!
* **Analogy:** A Russian nesting doll. You open the `address` doll, and inside you find a `city` doll and a `zip` doll.
* **In our schema:** The `address` column is a `StructType` containing `city` and `zip`.

### 🗺️ 3. `MapType` (Dictionaries / Key-Value Pairs)
A Map is a collection of Key-Value pairs, similar to a Python Dictionary. 
* **Analogy:** A contact book in your phone. You look up a Key (the person's name) to find the Value (their phone number).

<img src="../assets/Module_10/10_13_01.png" alt="Diagram illustrating the complex data types. ArrayType is shown as a shopping cart with multiple items. StructType is shown as a Russian Nesting Doll. MapType is shown as a Rolodex/Contact book." width="75%">

## 2. Accessing Nested Fields inside a `StructType`

If a Data Scientist asks you, *"Can you give me a table of users and just their cities?"*, how do you do it?

If you try `df.select("city")`, Spark will crash! It will say: *"Column 'city' does not exist!"* 
Why? Because `city` is trapped inside the `address` Struct.

**The Solution:** We use **Dot Notation**. We tell Spark to look inside the struct using `F.col("struct_name.field_name")`.

In [ ]:
print("--- Extracting Nested Struct Fields ---")

# Let's "flatten" the address struct by pulling out the city and zip code
# into their own top-level columns.

df_flat_address = df_nested.select(
    F.col("user_id"),
    F.col("name"),
    F.col("address.city").alias("city"),  # Reaching inside the struct!
    F.col("address.zip").alias("zip_code")
)

df_flat_address.show()
df_flat_address.printSchema()
print("Notice how 'city' and 'zip_code' are now flat, top-level columns. Much easier to analyze!")

## 3. Unpacking Arrays: `explode()`

Structs are easy to unpack using dot notation. But what about the `purchases` column? It's an Array (a list).

If Alice bought `["Laptop", "Mouse", "Monitor"]`, and a Data Scientist wants to group by product and count total sales, they can't! The items are trapped together in a single row.

**The Solution:** The `explode()` function.
`explode()` takes an Array and literally "explodes" it, creating a brand new row for every single item in the array, while duplicating the surrounding data (like `user_id` and `name`).

<img src="../assets/Module_10/10_13_02.png" alt="A visual showing a single row for Alice with a box of 3 items (Laptop, Mouse, Monitor). It passes through a dynamite icon labeled 'explode()', and outputs 3 separate rows for Alice, one for each item." width="75%">

In [ ]:
print("--- Exploding Arrays into Rows ---")

# We use explode() inside a select() or withColumn() to break the list apart.
df_exploded = df_nested.select(
    F.col("user_id"),
    F.col("name"),
    F.explode(F.col("purchases")).alias("single_purchase") # The magic function!
)

print("Before Explode (3 rows total):")
df_nested.select("user_id", "name", "purchases").show()

print("After Explode (6 rows total! Alice got split into 3 rows):")
df_exploded.show()

print("Now an Analyst can easily run a SQL GROUP BY on 'single_purchase'!")

## 4. Tracking Position: `posexplode()`

Sometimes, the *order* of the items in the list matters. 
For example, maybe the first item in the `purchases` array was their first purchase of the year, and the second item was their second purchase.

If you just use `explode()`, you lose that ranking information. To solve this, we use `posexplode()` (Position Explode). 

Instead of returning just one column (the item), `posexplode()` returns **two columns**: the index position (0, 1, 2...) and the item itself.

In [ ]:
print("--- Using posexplode() to keep the order ---")

# Because posexplode returns TWO columns, we use it inside a select() 
# and use .alias() to name both of the new columns.

df_pos_exploded = df_nested.select(
    F.col("name"),
    F.posexplode(F.col("purchases")).alias("purchase_order", "item_name")
)

df_pos_exploded.show()
print("Notice how we now know that Alice's 'Mouse' was purchase order #1 (the 2nd item)!")

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Handling complex JSON is a massive part of modern cloud architecture. 

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Data Ingestion** | Expects flat CSV files to load into SQL tables. Fails if handed a nested JSON document. | **Ingests massive, highly nested JSON logs from modern web APIs into a Data Lake using Spark.** | Expects a flat Pandas DataFrame. Gets frustrated by nested JSON lists. |
| **Data Flattening** | Normalizes data into multiple tables (e.g., a Users table and a separate Purchases table). | **Uses `explode()` and `col("struct.field")` to "flatten" the JSON into a highly denormalized, wide analytics table.** | Depends entirely on the DE to flatten the data before training a model. |
| **Ecosystem Location** | OLTP (Online Transaction Processing) Database. | **The ETL (Extract, Transform, Load) Pipeline.** | The Modeling Environment. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Relies on third-party tools to automatically "flatten" JSON files before Spark even sees them. Struggles when those tools break on highly complex API responses.
2. **Mid-Level DE (Your Current Level):** Comfortably navigates `StructTypes` and uses `explode()` to parse complex web logs, IoT sensor data, and API responses directly in PySpark.
3. **Senior DE:** Builds automated "Schema Parsers." A Senior DE can write PySpark code that recursively scans an unknown JSON file with 50 levels of nesting, figures out the schema, and writes a dynamic script to flatten the entire thing automatically.

### 🛠️ Where Data Engineering Fits in Modern Organizations
Software Engineers build apps that generate nested JSON. Data Scientists build AI models that require flat rows and columns. **Data Engineers are the bridge.** The techniques in this lesson (`explode` and dot notation) are exactly how you build that bridge.

## 🔑 Key Takeaways

- **StructType:** A nested object (like a dictionary inside a column). Access its internal fields using dot notation: `F.col("column.sub_column")`.
- **ArrayType:** A list of items in a single cell. You cannot easily group or aggregate data trapped in a list.
- **`explode()`:** Takes an Array column and "explodes" it, creating a new row for every single item in the list, duplicating the other columns.
- **`posexplode()`:** Exactly like explode, but it returns two columns: the index (position) of the item, and the item itself.
- **"Flattening" Data:** The process of taking nested JSON (Structs and Arrays) and turning it into a flat, 2D table that analysts can easily query using standard SQL.

## 📝 Knowledge Check Quiz

**Question 1:** You are reading a JSON file containing user profiles. The `location` column is a `StructType` containing `country`, `state`, and `city`. How do you extract just the `city` into a new column?
A) `df.select(F.explode("location"))`
B) `df.select(F.col("location.city"))`
C) `df.select(F.split("location", "city"))`
D) `df.select(F.col("city"))`

**Question 2:** A DataFrame has 10 rows. One of the columns is an `ArrayType` (a list). Every list contains exactly 5 items. If you use `F.explode()` on that array column, how many rows will the resulting DataFrame have?
A) 10 rows
B) 15 rows
C) 50 rows
D) 5 rows

**Question 3:** What is the main difference between `explode()` and `posexplode()`?
A) `explode()` only works on Structs, while `posexplode()` works on Arrays.
B) `posexplode()` only explodes positive numbers.
C) `explode()` returns one column (the item), while `posexplode()` returns two columns (the index position of the item, and the item itself).
D) There is no difference; they are the exact same function.

---

*Answers: 1: B, 2: C (10 rows * 5 items each = 50 separate rows), 3: C*

In [ ]:
# Clean up our session
spark.stop()
if os.path.exists("nested_api_data.json"):
    os.remove("nested_api_data.json")
print("Spark session closed and files cleaned up.")

### 🚀 What's Next?
We have learned how to clean strings, handle dates, and unpack nested JSON objects! Our data is almost ready for analytics.

But there is one massive, hidden danger left in our raw data: **Missing Data**. 

What happens when a cell is completely blank? If you try to do math on a blank cell, Spark might crash! In the next lesson, **10.14 Handling Nulls & Missing Data**, we will learn how to safely drop, fill, and manage Null values. See you there!